This notebook is based on one of Raschke's notebooks which accompanies 

**Building a Large Language Model**.  2025.  Raschka, Sebastian.

##  Exploring GPT2

[OpenAI Team on Limitations and Bias.](https://huggingface.co/openai-community/gpt2)

In [1]:
import sys
#dir(sys)
sys.version_info

sys.version_info(major=3, minor=12, micro=5, releaselevel='final', serial=0)

Also for using Hugging Face Models, check that we have authorization data in the environment (prviding a key downloaded from Huggingface).  This  is optional but prevent some warning messages.

In [2]:
import os
HF_TOKEN = os.environ["HF_TOKEN"]

In [3]:
from importlib.metadata import version

pkgs = ["matplotlib",  # Plotting library
        "numpy",       # PyTorch & TensorFlow dependency
        "tiktoken",    # Tokenizer
        "torch",       # Deep learning library
        "tensorflow",  # For OpenAI's pretrained weights
        "pandas"       # Dataset loading
       ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

matplotlib version: 3.10.0
numpy version: 2.4.2
tiktoken version: 0.12.0
torch version: 2.5.1
tensorflow version: 2.21.0
pandas version: 2.2.3


## Creating an untrained GPT langiage model

Define basic GPT-2 architecture (from Raschka, Ch. 6).

Obligatory to execute this.

In [31]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    """
    x_out = self.forward(x_in)
    
    x_in: batch_size x context_length x d_in 
    
    x_out: batch_size x context_length x d_out
    
    Note that an instantiated learner is indifferent to 
    the particular value of batch_size. The assumption is
    simply that the input is a 3D matrix and the first dimension is batch
    size, but only context_length, d_in, and d_out are parameters of 
    the learner.
    
    The d_out dimensions will be evenly divided among the heads, so
    d_out must be evenly divisible by num_heads. 
    """
    def __init__(self, d_in, d_out, 
                 context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        # Will divide the d_out dimensions evenly among the heads
        self.head_dim = d_out // num_heads    #1
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)   
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, context_length, d_in = x.shape
        # keys is now b x context_length x d_out
        keys = self.W_key(x)         #3
        queries = self.W_query(x)    #3
        values = self.W_value(x)     #3

        # dividing d_out into num_heads components; each component has shape head_dim 
        # keys is now b x context_length x num_heads x head_dim
        keys = keys.view(b, context_length, self.num_heads, self.head_dim)       #4
        values = values.view(b, context_length, self.num_heads, self.head_dim)  
        queries = queries.view(                                             
            b, context_length, self.num_heads, self.head_dim                    
        )                                                                   


        # keys becomes b x num_heads x context_length x head_dim
        keys = keys.transpose(1, 2)          #5
        queries = queries.transpose(1, 2)    #5
        values = values.transpose(1, 2)      #5

        # make keys compatible for matrix multiplication 
        #   mapping     @  mapped
        #   (..., m, n) @ (..., n, m)
        # keys becomes b x num_heads x head_dim X context_length
        # attn_scores will be b x num_heads x context_length x context_length
        attn_scores = queries @ keys.transpose(2, 3)   #6
        mask_bool = self.mask.bool()[:context_length, :context_length]    #7

        attn_scores.masked_fill_(mask_bool, -torch.inf)     #8

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # attn_weights : b x num_heads x context_length x context_length
        # values :  b x num_heads x context_length x head_dim
        # context_vec:  b x num_heads x context_length x head_dim
        # transpose(1, 2) =>
        # context_vec:  b x context_length x num_heads x head_dim
        context_vec = (attn_weights @ values).transpose(1, 2)   #9
        # Recombine num_heads x head_dim into d_out
        # context_vec:  b x context_length x d_out
        context_vec = context_vec.contiguous().view(
            b, context_length, self.d_out
        )                                           #10
        # And one more linear map for good luck to produce what we might call logits;
        # 2D map is a square matrix:
        # context_vec:  b x context_length x d_out
        context_vec = self.out_proj(context_vec)    #11
        return context_vec       

class LayerNorm(nn.Module):
    """
    
    """
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

# We wrap it in a feed forward layer
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)
    
class TransformerBlock(nn.Module):
    """
    Required input argument is a GPT 
    configuration dictionary (GPT_CONFIG_124M in this NB).
    """
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x  #1
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut      #2

        shortcut = x         #3
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut      #4
        return x
#1 Shortcut connection for attention block
#2 Add the original input back
#3 Shortcut connection for feed forward block
#4 Adds the original input back

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )
        
    
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
 #1
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

try:
    from accelerate import Accelerator
    use_accelerator=True
except:
    print("accelerator unavailble")
    use_accelerator=False
    

import os.path

def initialize_GPT_model(file_name, config,use_accelerator=use_accelerator):
    """
    Assumption this works for bioth large and small models.
    
    Distinctions encoded in config. and file_name, which must agree on model size.
    """
    model = GPTModel(config)
    device= choose_device(use_accelerator=use_accelerator)
    # Load locally save weights
    model.load_state_dict(torch.load(file_name, weights_only=True))
    model.to(device)
    return model

def initialize_GPT_model_no_wts(config,use_accelerator=use_accelerator):
    model = GPTModel(config)
    device= choose_device(use_accelerator=use_accelerator)
    model.to(device)
    return model

def initialize_GPTCLF_model_no_wts(config,use_accelerator=use_accelerator):
    model = GPTModelCLF(config)
    device= choose_device(use_accelerator=use_accelerator)
    model.to(device)
    return model

#1 The device setting will allow us to train 
#  the model on a CPU or GPU, depending on which device the input data sits on.
# Currently on my Mac, to get a qualifying torch version O need to use my transformers
# environment

# def choose_device():
#    if torch.cuda.is_available():
#        device = torch.device("cuda")
#    elif torch.backends.mps.is_available():
#        # Use PyTorch 2.9 or newer for stable mps results
#        major, minor = map(int, torch.__version__.split(".")[:2])
#        if (major, minor) >= (2, 9):
#            device = torch.device("mps")
#        else:
#            device = torch.device("cpu")
#    else:
#        device = torch.device("cpu")
#    return device

def choose_device (use_accelerator=False):
    if use_accelerator:
        return Accelerator().device
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        # Use PyTorch 2.9 or newer for stable mps results
        major, minor = map(int, torch.__version__.split(".")[:2])
        if (major, minor) >= (2, 9):
            device = torch.device("mps")
        else:
            device = torch.device("cpu")
    else:
        device = torch.device("cpu")
    return device

In [32]:
import torch

print("torch version:", version("torch"))

torch version: 2.5.1


Delete the following cell, eventually.

In [24]:
#from previous_chapters import GPTModel
# If the `previous_chapters.py` file is not available locally,
# you can import it from the `llms-from-scratch` PyPI package.
# For details, see: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# E.g.,
# from llms_from_scratch.ch04 import GPTModel

#GPT_CONFIG_124M = {
#    "vocab_size": 50257,   # Vocabulary size
#    "context_length": 256, # Shortened context length (orig: 1024)
#    "emb_dim": 768,        # Embedding dimension
#    "n_heads": 12,         # Number of attention heads
#    "n_layers": 12,        # Number of layers
#    "drop_rate": 0.1,      # Dropout rate
#    "qkv_bias": False      # Query-key-value bias
#}

#gpt = GPT_CONFIG_124M



#device=choose_device()
#print("Device:", device)

#torch.manual_seed(123)
#model = GPTModel(GPT_CONFIG_124M)
#model.eval();  # Disable dropout during inference

torch version: 2.5.1
Device: cpu


Execute this next cell at your own risk.  It overrules the conservative device choice made in the cell above.

On a Mac (per Raschke) the mps may not be numerically stable with
vcertain versions of `torch` (see below).

#  Using a GPT-2 pretrained model


This section shd be skipped at present.

This section shows how to 

1. Loads a lightweight version of the saved GPT2 weights saved to the local machine.
2. Download ublicly distributed GPT2 weights 


#### Set path to saved weights 

Weights were downloaded and saved in  the `weight-loading-pytorch.ipynb` notebook.

In [26]:
BASE_CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "drop_rate": 0.0,       # Dropout rate
    "qkv_bias": True        # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}


#CHOOSE_MODEL = "gpt2-small (124M)"
CHOOSE_MODEL = "gpt2-large (774M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

Delete the following?

In [69]:
import os.path

# Defined above
#root_dir = "/Users/gawron/Desktop/src/sphinx/python_for_ss_extras/colab_notebooks/"
# relative path from root
#data_dir = "transformers/HuggingFaceTransformers/raschke/LLMs-from-scratch/"
# Absolute path from root to data dir where saved models end up.
#data_dir = os.path.join(root_dir,data_dir)
################
#weights_dir = 'ch05/02_alternative_weight_loading'
#weights_dir  = os.path.join(data_dir, weights_dir)
#file_name = "gpt2-small-124M.pth"
#file_name = os.path.join(weights_dir,file_name)

## Downloading the Open AI model (under construction)

The default weights loaded are those for the large GPT 2 model.

These weights need a  `GPTModel(GPT_CONFIG_XXXM)` instance to load into.
Language model comparison: The perplexity on the Wiki data is almost exactly the same as the above
small GPT2 model, 29.66.

The download code needs both tensorflow and toprch.  There are issues with getting tensorflow and torch to play together in the same environment.  As a result I recommend installing tensorflow and torch into a special environment that you only use for executing the download code in this section.  Keep your wokring torch environment separate.  Save your mode

Do the download and then save the models and edit this notebook
to used the correct paths to the saved models. 

In [40]:
import os
import urllib.request
# import requests
import json
import numpy as np
import tensorflow as tf
from tqdm import tqdm

from tqdm import tqdm
import os.path

def download_and_load_gpt2(model_size, models_dir):
    # Validate model size
    allowed_sizes = ("124M", "355M", "774M", "1558M")
    if model_size not in allowed_sizes:
        raise ValueError(f"Model size not in {allowed_sizes}")

    # Define paths
    model_dir = os.path.join(models_dir, model_size)
    base_url = "https://openaipublic.blob.core.windows.net/gpt-2/models"
    backup_base_url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/gpt2"
    filenames = [
        "checkpoint", "encoder.json", "hparams.json",
        "model.ckpt.data-00000-of-00001", "model.ckpt.index",
        "model.ckpt.meta", "vocab.bpe"
    ]

    # Download files
    os.makedirs(model_dir, exist_ok=True)
    for filename in filenames:
        file_url = os.path.join(base_url, model_size, filename)
        backup_url = os.path.join(backup_base_url, model_size, filename)
        file_path = os.path.join(model_dir, filename)
        print(f"Downloading {file_url}/{file_path}")
        download_file(file_url, file_path, backup_url)
        print(f"Download complete")

    # Load settings and params
    tf_ckpt_path = tf.train.latest_checkpoint(model_dir)
    settings = json.load(open(os.path.join(model_dir, "hparams.json"), "r", encoding="utf-8"))
    params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)

    return settings, params


def download_file(url, destination, backup_url=None):
    def _attempt_download(download_url):
        with urllib.request.urlopen(download_url) as response:
            # Get the total file size from headers, defaulting to 0 if not present
            file_size = int(response.headers.get("Content-Length", 0))

            # Check if file exists and has the same size
            if os.path.exists(destination):
                file_size_local = os.path.getsize(destination)
                if file_size == file_size_local:
                    print(f"File already exists and is up-to-date: {destination}")
                    return True  # Indicate success without re-downloading

            block_size = 1024  # 1 Kilobyte

            # Initialize the progress bar with total file size
            progress_bar_description = os.path.basename(download_url)
            with tqdm(total=file_size, unit="iB", unit_scale=True, desc=progress_bar_description) as progress_bar:
                with open(destination, "wb") as file:
                    while True:
                        chunk = response.read(block_size)
                        if not chunk:
                            break
                        file.write(chunk)
                        progress_bar.update(len(chunk))
            return True

    try:
        if _attempt_download(url):
            return
    except (urllib.error.HTTPError, urllib.error.URLError):
        if backup_url is not None:
            print(f"Primary URL ({url}) failed. Attempting backup URL: {backup_url}")
            try:
                if _attempt_download(backup_url):
                    return
            except urllib.error.HTTPError:
                pass

        # If we reach here, both attempts have failed
        error_message = (
            f"Failed to download from both primary URL ({url})"
            f"{' and backup URL (' + backup_url + ')' if backup_url else ''}."
            "\nCheck your internet connection or the file availability.\n"
            "For help, visit: https://github.com/rasbt/LLMs-from-scratch/discussions/273"
        )
        print(error_message)
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        
def load_gpt2_params_from_tf_ckpt(ckpt_path, settings):
    # Initialize parameters dictionary with empty blocks for each layer
    params = {"blocks": [{} for _ in range(settings["n_layer"])]}

    # Iterate over each variable in the checkpoint
    for name, _ in tf.train.list_variables(ckpt_path):
        # Load the variable and remove singleton dimensions
        variable_array = np.squeeze(tf.train.load_variable(ckpt_path, name))

        # Process the variable name to extract relevant parts
        variable_name_parts = name.split("/")[1:]  # Skip the 'model/' prefix

        # Identify the target dictionary for the variable
        target_dict = params
        if variable_name_parts[0].startswith("h"):
            layer_number = int(variable_name_parts[0][1:])
            target_dict = params["blocks"][layer_number]

        # Recursively access or create nested dictionaries
        for key in variable_name_parts[1:-1]:
            target_dict = target_dict.setdefault(key, {})

        # Assign the variable array to the last key
        last_key = variable_name_parts[-1]
        target_dict[last_key] = variable_array

    return params

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])


ModuleNotFoundError: No module named 'tensorflow'

In [41]:
import os.path

#CHOOSE_MODEL = "gpt2-small (124M)"
CHOOSE_MODEL = "gpt2-large (774M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
models_dir ="gpt2"
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
model_dir = os.path.join(models_dir, model_size)
path = os.path.join(model_dir,"model.pt")

In [43]:
settings, params = download_and_load_gpt2(model_size=model_size, models_dir=models_dir)
#model = GPTModel(BASE_CONFIG)

NameError: name 'download_and_load_gpt2' is not defined

In [42]:
len(settings)

NameError: name 'settings' is not defined

In [12]:
#tf_ckpt_path = os.path.join(model_dir, "model.ckpt.data-00000-of-00001")
tf_ckpt_path = os.path.join(model_dir, "model.ckpt")
settings = json.load(open(os.path.join(model_dir, "hparams.json"), "r", encoding="utf-8"))
params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)

In [13]:
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

In [44]:
import os.path

def save_model (model, optimizer,epoch, loss, lr, path):
    torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
            'loss': loss,
            'lr':  lr
            }, path)

#models_dir,model_size="gpt2","124M"
model_dir = os.path.join(models_dir, model_size)
save_model(model,optimizer=None,epoch=0,loss=0,lr=5e-5,path=os.path.join(model_dir,"model.pt"))

#model = load_large_midel()

## The Cornell movie review dataset

In [4]:
from datasets import load_dataset_builder,get_dataset_split_names
ds_builder = load_dataset_builder("cornell-movie-review-data/rotten_tomatoes")

Movie Review Dataset. 
This is a dataset of containing 5,331 positive and 5,331 negative processed sentences from Rotten Tomatoes movie reviews. This data was first used in Bo Pang and Lillian Lee, ``Seeing stars: Exploiting class relationships for sentiment categorization with respect to rating scales.'', Proceedings of the ACL, 2005.

In [5]:
# Inspect dataset features
ds_builder.info.features
#{'label': ClassLabel(names=['neg', 'pos']),
# 'text': Value('string')}

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

In [6]:
get_dataset_split_names("cornell-movie-review-data/rotten_tomatoes")

['train', 'validation', 'test']

Load the actual data (you may have it saved locally -- to data_dir -- or load it from the HF site).

In [8]:
root_dir = "/Users/gawron/Desktop/src/sphinx/python_for_ss_extras/colab_notebooks/"
# relative path from root
data_dir0 = "transformers/HuggingFaceTransformers/raschke/LLMs-from-scratch/"
data_dir = os.path.join(root_dir,data_dir0)

In [10]:
data_dir

'/Users/gawron/Desktop/src/sphinx/python_for_ss_extras/colab_notebooks/transformers/HuggingFaceTransformers/raschke/LLMs-from-scratch/'

In [11]:
from datasets import load_dataset
from datasets import load_from_disk
import os.path
import os

HF_TOKEN = os.environ["HF_TOKEN"]

#data_dir = "/Users/gawron/Desktop/src/sphinx/"\
#"python_for_ss_extras/colab_notebooks/transformers/HuggingFaceTransformers/"\
#"raschke/LLMs-from-scratch/"
corpus_dir = os.path.join(data_dir,"ch06/01_main-chapter-code/")
dataset_nm = "cornell-movie-review"
dataset_nm = f"{dataset_nm}-all"
# For download from HF
dataset_huggingface_repo = "cornell-movie-review-data/rotten_tomatoes"

In [12]:
if os.path.isdir(os.path.join(corpus_dir,dataset_nm)):
    print("Load local")
    mr_data = load_from_disk(os.path.join(corpus_dir,dataset_nm))
else:
    mr_data = load_dataset(dataset_huggingface_repo,token=HF_TOKEN)
    # save local copy for next time
    mr_data.save_to_disk(dataset_nm)

Load local


In [13]:
mr_train = mr_data["train"]
mr_val = mr_data["validation"]
mr_test = mr_data["test"]

## Dataset specific code

In [14]:
#from chapter02 import create_dataloader_v1
from torch.utils.data import Dataset, DataLoader
from datasets.arrow_dataset import Dataset as ArrowDataset
import time
import pandas as pd

max_length,stride = 256, 128

class MovieReviewDataset(Dataset):
    #class SpamDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=None, pad_token_id=50256):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # Truncate sequences if they are longer than max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # Pad sequences to the longest sequence
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data[index]["label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length
        # Note: A more pythonic version to implement this method
        # is the following, which is also used in the next chapter:
        # return max(len(encoded_text) for encoded_text in self.encoded_texts)    

def get_cross_validation_splits(train,val,test,nfolds=3,seed=47,
                               ):
    """
    train,val,test are Dataset instances (e.g., datasets.arrow_dataset.Dataset).
    
    Merge data, split into nfolds pieces, create train,val,test loaders
    for each fold.  
    
    this is a generator.
    """
    items = (train, val, test)
    df = pd.concat([item.to_pandas() for item in items])
    #Idiom for shuffling data
    df = df.sample(frac=1,random_state=seed).reset_index(drop=True)
    fold_len = len(df)//nfolds
    dfs = [df.iloc[i*fold_len:(i*fold_len)+fold_len] for i in range(nfolds)]
    #len of val,test sets assuming equal size
    split_len = len(dfs[0])//2
    for i in range(nfolds):
        val_test = dfs[i]
        val, test = [val_test.iloc[(i*split_len):((i*split_len)+split_len)] for i in range(2)]
        val, test = ArrowDataset.from_pandas(val),ArrowDataset.from_pandas(test)
        train = pd.concat(dfs[:i] + dfs[i+1:])
        train = ArrowDataset.from_pandas(train)
        yield (train, val, test)
        
def run_mr_cross_vals(mr_train, mr_val, mr_test, model_file=None, nfolds=3,seed=123,
                      batch_size=20, num_workers=0,num_epochs=8,
                     lr=5e-5,eval_freq=50,eval_iter=5,
                     save_path="model.pt"):
    """
    If model file is not None must pass in full path.
    """
    global results,mr_test_loader
    results= []
    for (i,(train, val, test)) in enumerate(
                                        get_cross_validation_splits(mr_train,mr_val,
                                                          mr_test,nfolds=nfolds,
                                                          seed=seed)
    ):

        mr_train_loader = DataLoader(
            dataset=MovieReviewDataset(train,tokenizer),
            batch_size=batch_size,
            drop_last=True,
            shuffle=True,
            num_workers=num_workers
        )
        mr_val_loader = DataLoader(
            MovieReviewDataset(val,tokenizer),
            batch_size=batch_size,
            drop_last=False,
            shuffle=False,
            num_workers=num_workers
        )
        mr_test_loader = DataLoader(
            MovieReviewDataset(test,tokenizer),
            batch_size=batch_size,
            drop_last=False,
            shuffle=False,
            num_workers=num_workers
        )
        result = run_training (mr_train_loader,  mr_val_loader, 
                               model_file=model_file, num_epochs=num_epochs, 
                               lr=lr, eval_freq=eval_freq, eval_iter=eval_iter,
                               save_path=save_path,seed=seed)
        
        results.append(result)
        #test_accuracy0 = calc_accuracy_loader(data_loader, model, device)
        print(f"***Getting test results for validation run {i+1}!***")
        # fully trained model is the denotation of global variable `model`
        #OR eval_model=model,save_path=None
        result["test_acc"] = get_test_results(mr_test_loader,num_epochs,
                                              save_path=save_path,seed=seed)
        print("***End Testing!***")
    avg_acc = sum([result["test_acc"] for result in results])/nfolds
    print(f"Average accuracy over {nfolds}: {avg_acc:.2f}")
    return avg_acc, results
 
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()

    # Prepare inputs to the model
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]
    # Note: In the book, this was originally written as pos_emb.weight.shape[1] by mistake
    # It didn't break the code but would have caused unnecessary truncation (to 768 instead of 1024)

    # Truncate sequences if they too long
    #input_ids = input_ids[:min(max_length, supported_context_length)]
    #assert max_length is not None, (
    #    "max_length must be specified. If you want to use the full model context, "
    #    "pass max_length=model.pos_emb.weight.shape[0]."
    #)

    
    #assert max_length <= supported_context_length, (
    #    f"max_length ({max_length}) exceeds model's supported context length ({supported_context_length})."
    #)    
    # The following robust version is Raschka's alternative, which handles the max_length=None case better
    max_length = min(max_length,supported_context_length) if max_length else supported_context_length
    # Truncate sequences if they too long
    input_ids = input_ids[:min(max_length, supported_context_length)]
    
    # Pad sequences to the longest sequence
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) # add batch dimension

    # Model inference
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]  # Logits of the last output token
    predicted_label = torch.argmax(logits, dim=-1).item()

    # Return the classified result
    return predicted_label




### Make the tokenizer and the novie review data loaders, one loader for each split

In [15]:
import tiktoken
import numpy as np

tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))


def count_padding  (item):
    return  (item[0]==50256).sum()

[50256]


In [20]:
import torch 

torch.manual_seed(123)
num_workers = 0
batch_size = 20

torch.manual_seed(123)

mr_train_loader = DataLoader(
    dataset=MovieReviewDataset(mr_train,tokenizer),
    batch_size=batch_size,
    drop_last=True,
    shuffle=True,
    num_workers=num_workers
)

mr_val_loader = DataLoader(
    MovieReviewDataset(mr_val,tokenizer),
    batch_size=batch_size,
    drop_last=False,
    shuffle=False,
    num_workers=num_workers
)

mr_test_loader = DataLoader(
    MovieReviewDataset(mr_test,tokenizer),
    batch_size=batch_size,
    drop_last=False,
    shuffle=False,
    num_workers=num_workers
)


## Classifier training code, classifier

A lot of this is similar to Raschka's training code for train the large language models,
by Raschka's design.

In [37]:
# Overall the same as `train_model_simple` in chapter 5
# JMG added save best model feature
from torch import inf
import time

def run_training (train_loader, val_loader, model_file=None, num_epochs=8, lr=5e-5, 
                   eval_freq=50,eval_iter=5,save_path="model.pt",seed=123,
                  use_accelerator=True):
    """
    Creates model and runs num_epochs of training.  
    
    Wrapper for train_classifier_simple.
    
    Alseo creates optimizer and a few other housekeeping bits
    and calls train_classifier_simple.
    
    If model_file is None, fetches the large GPT model as default.
    
    If model file is not None, must pass in full path, for example 
    the global wt_file_name, which loads the small GPT model.
    
    global device must also be set.
    
    Returns result dictionary.
    """
    global model
    start_time = time.time()
    torch.manual_seed(seed)
    #  Clear out some memory for a BIG object
    if "model" in globals().keys():
        del model
    # This loads the pretrained LLM and beheads it to make it a classifier
    model= make_classifier_model(path=model_file,use_accelerator=use_accelerator)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    train_losses, val_losses, train_accs, val_accs, examples_seen =\
      train_classifier_simple(
        model, train_loader, val_loader, optimizer, 
        num_epochs=num_epochs, eval_freq=eval_freq, eval_iter=eval_iter,
        save_path=save_path
        )
    end_time = time.time()
    execution_time_minutes = (end_time - start_time) / 60
    print(f"{num_epochs} epochs of training completed with {lr=}"\
          f" in {execution_time_minutes:.2f} minutes.")
    result = dict(train_losses=train_losses,
                 val_losses=val_losses,
                 train_accs=train_accs,
                 val_accs=val_accs)
    return result

def train_classifier_simple(model, train_loader, val_loader, optimizer, num_epochs,
                            eval_freq, eval_iter,save_path=None,accuracy_iter=None,
                            use_accelrator=True):
    # Initialize lists to track losses and examples seen
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step,best_accuracy = 0, -1, 0
    if accuracy_iter== None:
        accuracy_iter = 2 * eval_iter
    device = choose_device(use_accelerator=use_accelerator)
    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            examples_seen += input_batch.shape[0] # New: track examples instead of tokens
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Calculate accuracy after each epoch
        train_accuracy = calc_accuracy_loader(train_loader, model, device, 
                                              num_batches=accuracy_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, 
                                            num_batches=accuracy_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        # A small GPT model takes up 1.4 GB 
        if save_path is not None and val_accuracy > best_accuracy:
            lr = optimizer.param_groups[-1]["lr"]
            save_model (model, optimizer, epoch+1, val_loss, lr, save_path)
            best_accuracy = val_accuracy
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen


def load_model2 (model_file,gpt_config=None):
    if gpt_config is None:
        gpt_config = GPT_CONFIG_124M
    model, optimizer = initialize_model_and_optimizer(gpt_config=gpt_config,
                                                     use_optimizer=True)
    checkpoint = torch.load(model_file, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    # return epoch, loss, lr
    return checkpoint['epoch'], checkpoint['loss'], checkpoint['lr']

def load_classifier_model (path):
    """
    Load model downloaded from HuggingFace and saved locally
    in gpt2 dir.  Used on initializing training.
    
    Model size in global BASE_CONFIG via global CHOOSE_MODEL;
    e.g., we have executed:
    
    CHOOSE_MODEL = "gpt2-small (124M)"
    BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
    """

    # Initialize base model
    model = GPTModelCLF(BASE_CONFIG)
    # Convert model to classifier as in section 6.5 in ch06.ipynb
    #num_classes = 2
    #model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], 
    #                                 out_features=num_classes)
    # Find path to saved weights
    models_dir ="gpt2"
    model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
    model_dir = os.path.join(models_dir, model_size)
    path = os.path.join(model_dir,model_file)
    # Get a device
    device = choose_device(use_accelerator=use_accelerator)
    # Then load pretrained weights
    model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    model.to(device)
    model.eval();
    return model
    
def load_classifier_model2 (model_file):
    # We start with all the old weigths of GPT mopdel
    model= make_classifier_model(wt_file_name)
    #model = GPTModel(config)
    #device= choose_device(use_accelerator=use_accelerator)
    #model.to(device)
    # now we load on whatever training has given us.
    checkpoint = torch.load(model_file, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    # return epoch, loss, lr
    return model, checkpoint['epoch'], checkpoint['loss'], checkpoint['lr']

def save_model (model, optimizer,epoch, loss, lr, path):
    torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
            'loss': loss,
            'lr':  lr
            }, path)
    

def load_large_model (root_dir=None,data_dir=None,
                      models_dir="gpt2",use_accelerator=True):
    """
    Loads large GPT model.
    
    Note: no classifier out_head as yet.
    """
    if root_dir is None:
        root_dir=("/Users/gawron/Desktop/src/sphinx/python_for_ss_extras/colab_notebooks/"
                  "transformers/HuggingFaceTransformers/raschke/LLMs-from-scratch/")
        if data_dir is None:
            data_dir="ch06/01_main-chapter-code/"
        models_dir = os.path.join(root_dir, data_dir, models_dir)
    model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
    model_dir = os.path.join(models_dir, model_size)
    path = os.path.join(model_dir,"model.pt")
    print(path)
    device = choose_device(use_accelerator=use_accelerator)
    model = initialize_GPT_model(path, config=BASE_CONFIG,use_accelerator=use_accelerator)
    #model.load_state_dict(torch.load(path, weights_only=True))
    model.to(device)
    model.eval()
    return model

# Same as chapter 5
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]  # Logits of last output token
    #print(logits.shape,target_batch.shape)
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

# Same as in chapter 5
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches


def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # Logits of last output token
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

def calc_acc_prec_rec_loader(data_loader, model, device, num_batches=None):
    """
    Accuracy precision recall computed without importing metric functions.
    
    It would be a little cleaner to build predicted and trues sequences and then
    compute a,p,r using metrics functions.
    """
    model.eval()
    true_positives,num_examples,positives,trues = 0, 0, 0,0
    #predicted,trues = [],[]
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # Logits of last output token
            predicted_labels = torch.argmax(logits, dim=-1)
            true_positives += (predicted_labels == target_batch).sum().item()
            num_examples += predicted_labels.shape[0]
            positives += predicted_labels.sum().item()
            trues += target_batch.sum().item()
        else:
            break
    #return correct_predictions / num_examples
    print(target_batch.shape, predicted_labels.shape)
    accuracy = true_positives/num_examples if num_examples > 0 else 0
    precision = true_positives/positives  if positives > 0 else 0
    recall = true_positives/trues  if trues > 0 else 0
    return accuracy,precision,recall

   
def get_test_results(data_loader, n_epochs, eval_model=None, save_path=None,
                     seed=None, clobber=True):
    global model
    print("save_path",save_path)
    if save_path is not None:
        #  Clear out some memory for a BIG object model var shd be defined
        #  if not, this will raise an error
        if clobber and "model" in globals().keys():
            print(f"Test: Clobbering epoch {n_epochs} model.")
            del model
        model = load_saved_classifier_model (save_path)
        print(f"Using best model.")
    else:
        model = eval_model
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    with torch.no_grad():
        #test_loss = calc_loss_loader(data_loader, model, device)
        test_accuracy = calc_accuracy_loader(data_loader, model, device)
    print(f"Test accuracy:  {test_accuracy:.2%}")
    return test_accuracy

def make_empty_classifier_model(seed=123,num_classes=2,
                         use_accelerator=True, 
                         ):
    """
    This is the way Raschke does it in the text and so far this is
    the only way that works for reloading saved weights too.
    
    Using a class defined with the right Linear output (head.out)
    from the getgo doesnt work, and I don;t know why.
    """
    device = choose_device(use_accelerator=use_accelerator)
    torch.manual_seed(seed)
    model = initialize_GPT_model_no_wts(config=BASE_CONFIG,
                                    use_accelerator=use_accelerator)
    convert_to_classifier_model (model,device,num_classes=2)
    return model


def load_states(model, model_file="model.pt"):
    # Get a device
    device = choose_device(use_accelerator=use_accelerator)
    # Then load pretrained weights
    params = torch.load(model_file, map_location=device, weights_only=True)
    model.load_state_dict(params["model_state_dict"])
    model.to(device)
    model.eval();

def load_saved_classifier_model (model_file="model.pt",num_classes=2):
    model = make_empty_classifier_model(num_classes=num_classes)
    load_states(model, model_file=model_file)
    return model

def old_make_classifier_model(path, seed=123,num_classes=2, use_accelerator=True, 
                          models_dir="gpt2"
                          ):
    """
    This one initializes with wts, which is unnecessary on reloading
    saved weights.
    """
    if path is None:
        path = get_model_file_name(models_dir=models_dir)
    device = choose_device(use_accelerator=use_accelerator)
    torch.manual_seed(123)
    model = initialize_GPT_model(path, 
                                config=BASE_CONFIG,
                                use_accelerator=use_accelerator)
    convert_to_classifier_model (model,device, num_classes=num_classes)
    return model

def make_classifier_model(path, seed=123,num_classes=2, use_accelerator=True, 
                          models_dir="gpt2",
                          ):
    """
    This one initializes with wts, which is unnecessary on reloading
    saved weights.
    """
    if path is None:
        path = get_model_file_name(models_dir=models_dir)
    torch.manual_seed(123)
    model = initialize_GPT_model(path, 
                                config=BASE_CONFIG,
                                use_accelerator=use_accelerator)
    convert_to_classifier_model (model,use_accelerator=use_accelerator,num_classes=num_classes)
    return model

def get_model_file_name(models_dir="gpt2",root_dir=None,data_dir=None):
    if root_dir is None:
        root_dir=("/Users/gawron/Desktop/src/sphinx/python_for_ss_extras/colab_notebooks/"
                  "transformers/HuggingFaceTransformers/raschke/LLMs-from-scratch/")
    if data_dir is None:
        data_dir="ch06/01_main-chapter-code/"
    models_dir = os.path.join(root_dir, data_dir, models_dir)
    model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
    model_dir = os.path.join(models_dir, model_size)
    return os.path.join(model_dir,"model.pt")

def convert_to_classifier_model (model,use_accelerator=True,num_classes=2):
    device= choose_device(use_accelerator=use_accelerator)
    model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], 
                                     out_features=num_classes).to(device)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.trf_blocks[-1].parameters():
        param.requires_grad = True
    for param in model.final_norm.parameters():
        param.requires_grad = True    

def old_get_model_file_name(models_dir="gpt2"):
    model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
    #model_size='774M'
    model_dir = os.path.join(models_dir, model_size)
    return os.path.join(model_dir,"model.pt")


    
###############################################################
####   Retired (with misgivings)
###############################################################

def make_empty_classifier_model2(seed=123,num_classes=2,
                         use_accelerator=True, 
                         ):
    """
    This version doesnt work.  No idea why.
    
    Something wrong with the class definition?
    """
    #device = choose_device(use_accelerator=use_accelerator)
    torch.manual_seed(seed)
    #initialize_GPTCLF_model_no_wts
    model = GPTModelCLF(BASE_CONFIG)
    return model

Train the movie review classifier

In [36]:
result = run_training (mr_train_loader, mr_val_loader, num_epochs=5)

Ep 1 (Step 000000): Train loss 0.688, Val loss 0.603
Ep 1 (Step 000050): Train loss 0.725, Val loss 0.534
Ep 1 (Step 000100): Train loss 0.675, Val loss 0.662
Ep 1 (Step 000150): Train loss 0.611, Val loss 0.702
Ep 1 (Step 000200): Train loss 0.494, Val loss 0.235
Ep 1 (Step 000250): Train loss 0.322, Val loss 0.358
Ep 1 (Step 000300): Train loss 0.328, Val loss 0.269
Ep 1 (Step 000350): Train loss 0.396, Val loss 0.142
Ep 1 (Step 000400): Train loss 0.322, Val loss 0.242
Training accuracy: 85.50% | Validation accuracy: 88.50%
Ep 2 (Step 000450): Train loss 0.331, Val loss 0.139
Ep 2 (Step 000500): Train loss 0.246, Val loss 0.204
Ep 2 (Step 000550): Train loss 0.248, Val loss 0.206
Ep 2 (Step 000600): Train loss 0.227, Val loss 0.157
Ep 2 (Step 000650): Train loss 0.362, Val loss 0.117
Ep 2 (Step 000700): Train loss 0.306, Val loss 0.179
Ep 2 (Step 000750): Train loss 0.343, Val loss 0.164
Ep 2 (Step 000800): Train loss 0.198, Val loss 0.147
Ep 2 (Step 000850): Train loss 0.330, Val l

In [39]:
device=choose_device(use_accelerator=True)
test_accuracy = calc_accuracy_loader(mr_test_loader, model, device)
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Test accuracy: 86.59%


### A bare bones view of what we did above

In [ ]:
import time
  
    
model= make_classifier_model()
lr=5e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)

start_time = time.time()
num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, mr_train_loader, mr_val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Test on your favorite review.

In [201]:
text_2 = (
    "Inception is the greatest thing since sliced avocados"
)

print(classify_review(
    text_2, model, tokenizer, device))

1


##  Cross validation

In [ ]:
num_epochs,nfolds,seed = 4, 3, 123
run_mr_cross_vals(mr_train, mr_val, mr_test, nfolds=nfolds,seed=seed)